# Naive Bayes

Imports and helpers

In [ ]:
import scipy.sparse as sp
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.naive_bayes import MultinomialNB
from scipy.stats import uniform
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
imprt joblib
import numpy as np

back to root to call functions from helpers.py file

In [2]:
import os
from pathlib import Path
print(Path.cwd())
os.chdir(Path('..').resolve())
from src.utils.helpers import save
print(Path.cwd())

/mnt/d/career/digilians/sentiment-analysis-of-amazon-reviews-using-machine-learning/notebooks
/mnt/d/career/digilians/sentiment-analysis-of-amazon-reviews-using-machine-learning


## Data Acquisition

load dataset

In [3]:
X_train = sp.load_npz('data/vectorizers/X_train_tfidf.npz')
print(f"X_train shape: {X_train.shape}")
y_train = pd.read_csv('data/processed/y_train.csv').squeeze()
print(f"y_train shape: {y_train.shape}")

X_train shape: (79972, 20000)
y_train shape: (79972,)


In [4]:
X_valid = sp.load_npz('data/vectorizers/X_valid_tfidf.npz')
print(f"X_valid shape: {X_valid.shape}")
y_valid = pd.read_csv('data/processed/y_valid.csv')
print(f"y_valid shape: {y_valid.shape}")

X_valid shape: (20000, 20000)
y_valid shape: (20000, 1)


In [5]:
X_test = sp.load_npz('data/vectorizers/X_test_tfidf.npz')
print(f"X_test shape: {X_test.shape}")
y_test = pd.read_csv('data/processed/y_test.csv').squeeze()
print(f"y_test shape: {y_test.shape}")

X_test shape: (20000, 20000)
y_test shape: (20000,)


## Model Training

model setup and random search 

In [6]:
nb_model = MultinomialNB()
nb_param_dist = {'alpha': uniform(0.001, 4.0), 'fit_prior': [True, False]}

Initialize RandomizedSearchCV (3-fold cross-validation acts as our validation set)

In [7]:
nb_search = RandomizedSearchCV(nb_model, nb_param_dist, 
                               n_iter=60, cv=5, scoring='accuracy', 
                               random_state=42, n_jobs=-1, verbose=1)

In [8]:
print("\nTraining Naive Bayes...")
start_time = time.time()
nb_search.fit(X_train, y_train)
nb_train_time = time.time() - start_time
print(f"Completed in {nb_train_time:.2f}s. Best params: {nb_search.best_params_}")


Training Naive Bayes...
Fitting 5 folds for each of 60 candidates, totalling 300 fits


Completed in 18.39s. Best params: {'alpha': np.float64(1.4348629141770903), 'fit_prior': True}


Extract best models

In [9]:
best_nb = nb_search.best_estimator_
print(f"Completed in {nb_train_time:.2f}s.")
print(f"Best NB Params: {nb_search.best_params_}")

Completed in 18.39s.
Best NB Params: {'alpha': np.float64(1.4348629141770903), 'fit_prior': True}


## Model Evaluation

Predict on the X_valid 

In [ ]:
y_valid_pred = best_nb.predict(X_valid)
print(f"Validation Accuracy: {accuracy_score(y_valid, y_valid_pred):.4f}")

confusion matrices

In [ ]:
print(f"Training Time: {nb_train_time:.4f} seconds")
# Ensure y_valid is a 1D series
print("\nClassification Report:")
print(classification_report(y_valid, y_valid_pred, target_names=['Negative', 'Positive']))

Training Time: 11.9559 seconds
              precision    recall  f1-score   support

    Negative       0.83      0.82      0.82      9966
    Positive       0.82      0.83      0.83     10034

    accuracy                           0.83     20000
   macro avg       0.83      0.83      0.83     20000
weighted avg       0.83      0.83      0.83     20000



## Model Testing

In [ ]:
y_test_pred = best_nb.predict(X_test)
print(f"Test Accuracy: {accuracy_score(y_test, y_test_pred):.4f}")

In [ ]:
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, target_names=['Negative', 'Positive']))

## Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
classes = best_nb.classes_
ticklabels = ['Negative', 'Positive']

sns.heatmap(confusion_matrix(y_valid, y_valid_pred, labels=classes), 
            annot=True, fmt='d', cmap='Blues', xticklabels=ticklabels, yticklabels=ticklabels, ax=axes[0])
axes[0].set_title('Validation Confusion Matrix')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

sns.heatmap(confusion_matrix(y_test, y_test_pred, labels=classes), 
            annot=True, fmt='d', cmap='Greens', xticklabels=ticklabels, yticklabels=ticklabels, ax=axes[1])
axes[1].set_title('Test Confusion Matrix')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')
plt.tight_layout()
plt.show()

## Feature Inspection & Saving

In [ ]:
vec_path = Path('data/vectorizers/tfidf_vectorizer.joblib')
if vec_path.exists():
    vectorizer = joblib.load(vec_path)
    feature_names = np.array(vectorizer.get_feature_names_out())
    
    # FIX: MultinomialNB uses feature_log_prob_, not coef_
    # feature_log_prob_[0] is negative class, [1] is positive class
    neg_class_prob_sorted = best_nb.feature_log_prob_[0, :].argsort()[::-1]
    pos_class_prob_sorted = best_nb.feature_log_prob_[1, :].argsort()[::-1]
    
    print('\nTop features indicating Positive sentiment:')
    for i in pos_class_prob_sorted[:15]: 
        print(f"{feature_names[i]:<15} {best_nb.feature_log_prob_[1, i]:.4f}")

    print('\nTop features indicating Negative sentiment:')
    for i in neg_class_prob_sorted[:15]: 
        print(f"{feature_names[i]:<15} {best_nb.feature_log_prob_[0, i]:.4f}")

## Saving the Model and Pipeline

In [ ]:
save(model_base='data/models', model=best_nb, model_name='best_naive_bayes.joblib')

## Insights

"""
**Insights - Naive Bayes:**
1. **Computational Speed:** As expected from our project proposal, Naive Bayes trained significantly faster than Logistic Regression. It relies on simple probability counts rather than iterative optimization (gradient descent).
2. **The Naive Assumption:** Despite the algorithm's assumption that every word's occurrence is independent of other words (which is false in human language), it still performs incredibly well as a baseline classifier.
3. **Laplace Smoothing ($\alpha$):** The hyperparameter search optimized our $\alpha$ value, ensuring that words present in the test set but absent in the training set did not result in a zero probability, stabilizing our predictions.
"""